In [ ]:
import pandas as pd
import numpy as np
from power_grid_model import (
    ValidationException,
    BatchUpdateDataset,
    BatchResult,
    power_flow,
    batch_power_flow,
)
from power_grid_model.io.serialization import deserialize_json
from power_grid_model.utils import get_element_ids_from_grid_data


class PowerGridCalculator:
    """
    A class to perform power grid calculations using the power-grid-model (PGM) library.

    This class handles loading grid data, creating batch update datasets from load profiles,
    running time-series power flow calculations, and aggregating the results into
    summary tables for node voltages and line losses/loadings.
    """

    def __init__(self, pgm_input_data: dict):
        """
        Initializes the PowerGridCalculator with PGM input data.

        The input data is used to construct the internal PGM grid object.

        Args:
            pgm_input_data (dict): A dictionary representing the power grid in PGM input format.

        Raises:
            ValidationException: If the pgm_input_data is invalid during PGM construction.
                                 This exception is passed through from the PGM library.
        """
        self._pgm_grid = None
        self._initialize_pgm(pgm_input_data)
        # Store base power for unit conversions. PGM typically uses MVA as base power.
        # Assuming base_power is available in the grid data, or a default of 1 MVA if not specified.
        # For simplicity, we'll assume a base power of 1 MVA if not explicitly found in grid data.
        # In a real scenario, this should be explicitly part of the PGM input or derived from it.
        # For now, let's assume 1 MVA base power for p.u. conversions to kWh.
        # If the PGM input data provides a base power, it should be used.
        # For now, we'll use a placeholder and note this assumption.
        # PGM's power_flow results are in p.u.
        # To convert p.u. power to actual power (MW or kW), we need the base power.
        # Let's assume a default base power of 1 MVA = 1000 kW for calculations if not specified.
        # A more robust solution would extract this from the PGM input data if available.
        # For this assignment, we'll use a common base power for typical distribution grids.
        # Let's assume a base power of 1 MVA for calculations.
        # 1 MVA = 1000 kVA. If power factor is 1, then 1 MW = 1000 kW.
        self._base_power_mva = 1.0 # Default base power in MVA

        # Attempt to extract base power from grid data if available
        if 'base_power_mva' in pgm_input_data:
            self._base_power_mva = pgm_input_data['base_power_mva']
        elif 'base_power' in pgm_input_data: # Sometimes it might be just 'base_power'
            self._base_power_mva = pgm_input_data['base_power'] / 1e6 # Convert from VA to MVA

        print(f"Initialized PowerGridCalculator with base power: {self._base_power_mva} MVA")


    def _initialize_pgm(self, pgm_input_data: dict):
        """
        Internal method to construct the PGM object from input data.

        This method uses `pgm.io.serialization.deserialize_json` to build the grid.
        It is called during the initialization of the PowerGridCalculator instance.

        Args:
            pgm_input_data (dict): The dictionary containing the PGM input data.

        Raises:
            ValidationException: Passed through if the input data is invalid.
        """
        try:
            self._pgm_grid = deserialize_json(pgm_input_data)
        except ValidationException as e:
            print(f"Error initializing PGM grid: {e}")
            raise  # Re-raise the exception to indicate failure

    def create_batch_update_dataset(
        self,
        active_load_profile: pd.DataFrame,
        reactive_load_profile: pd.DataFrame
    ) -> BatchUpdateDataset:
        """
        Creates a PGM batch update dataset from active and reactive load profiles.

        This method validates that the timestamps and load IDs match between the
        active and reactive load profiles before constructing the batch dataset.

        Args:
            active_load_profile (pd.DataFrame): DataFrame containing active load profiles.
                                                Expected format: columns 'timestamp', 'load_id', 'value'.
            reactive_load_profile (pd.DataFrame): DataFrame containing reactive load profiles.
                                                  Expected format: columns 'timestamp', 'load_id', 'value'.

        Returns:
            BatchUpdateDataset: The constructed PGM batch update dataset.

        Raises:
            ValueError: If the two profiles do not have matching timestamps and/or load IDs.
        """
        # Ensure 'timestamp' is in datetime format and set as index for easier comparison
        if 'timestamp' in active_load_profile.columns:
            active_load_profile['timestamp'] = pd.to_datetime(active_load_profile['timestamp'])
            active_load_profile = active_load_profile.set_index('timestamp')
        if 'timestamp' in reactive_load_profile.columns:
            reactive_load_profile['timestamp'] = pd.to_datetime(reactive_load_profile['timestamp'])
            reactive_load_profile = reactive_load_profile.set_index('timestamp')

        # Check for matching timestamps (index)
        if not active_load_profile.index.equals(reactive_load_profile.index):
            raise ValueError("Active and reactive load profiles do not have matching timestamps.")

        # Check for matching load IDs (assuming 'load_id' is a column that needs to be pivoted)
        # If the dataframes are already pivoted (timestamp as index, load_ids as columns),
        # then we compare columns.
        # Let's assume the input is in 'long' format and needs pivoting.
        # Pivot the dataframes to have timestamps as index and load_ids as columns
        p_profile_pivot = active_load_profile.pivot_table(index=active_load_profile.index, columns='load_id', values='value')
        q_profile_pivot = reactive_load_profile.pivot_table(index=reactive_load_profile.index, columns='load_id', values='value')

        if not p_profile_pivot.columns.equals(q_profile_pivot.columns):
            raise ValueError("Active and reactive load profiles do not have matching load IDs.")

        # Get the load IDs from the pivoted columns
        load_ids = p_profile_pivot.columns.tolist()
        timestamps = p_profile_pivot.index.tolist()

        # Create the list of update dictionaries for BatchUpdateDataset
        updates = []
        for ts in timestamps:
            timestamp_str = ts.isoformat() # Convert timestamp to ISO format string
            sym_load_updates = []
            for load_id in load_ids:
                p_value = p_profile_pivot.loc[ts, load_id]
                q_value = q_profile_pivot.loc[ts, load_id]
                sym_load_updates.append(
                    {"id": load_id, "p_mw": p_value, "q_mvar": q_value} # PGM expects MW/MVAR for sym_load
                )
            updates.append(
                {"timestamp": timestamp_str, "sym_load": sym_load_updates}
            )

        # Create BatchUpdateDataset
        batch_dataset = BatchUpdateDataset(updates)
        return batch_dataset

    def run_time_series_power_flow(
        self,
        batch_dataset: BatchUpdateDataset
    ) -> BatchResult:
        """
        Runs time-series (batch) power flow calculation.

        Args:
            batch_dataset (BatchUpdateDataset): The batch update dataset containing
                                                time-series load changes.

        Returns:
            BatchResult: The results of the time-series power flow calculation.

        Raises:
            ValidationException: If the batch_dataset is invalid. This exception
                                 is passed through from the PGM library.
        """
        if self._pgm_grid is None:
            raise RuntimeError("PGM grid not initialized. Call __init__ with valid PGM data first.")
        try:
            results = batch_power_flow(self._pgm_grid, batch_dataset)
            return results
        except ValidationException as e:
            print(f"Error running batch power flow: {e}")
            raise  # Re-raise the exception to indicate failure

    def aggregate_node_voltage_results(
        self,
        power_flow_results: BatchResult
    ) -> pd.DataFrame:
        """
        Aggregates node voltage results into a summary table.

        The table includes the maximum and minimum p.u. voltages for all nodes
        at each timestamp, along with the IDs of the corresponding nodes.

        Args:
            power_flow_results (BatchResult): The results from time-series power flow.

        Returns:
            pd.DataFrame: A table with aggregated voltage results.
                          Index: 'timestamp'
                          Columns: 'max_pu_voltage', 'max_voltage_node_id',
                                   'min_pu_voltage', 'min_voltage_node_id'.
        """
        # Extract node voltage magnitudes from results
        node_voltage_magnitudes = power_flow_results.get_data("node", "u_pu")
        # The data is typically a dictionary where keys are timestamps and values are dicts of node_id: u_pu

        summary_data = []
        for timestamp_str, node_data in node_voltage_magnitudes.items():
            u_pu_values = np.array(list(node_data.values()))
            node_ids = list(node_data.keys())

            if len(u_pu_values) == 0:
                continue

            max_pu_voltage = np.max(u_pu_values)
            min_pu_voltage = np.min(u_pu_values)

            max_voltage_node_id = node_ids[np.argmax(u_pu_values)]
            min_voltage_node_id = node_ids[np.argmin(u_pu_values)]

            summary_data.append({
                "timestamp": pd.to_datetime(timestamp_str),
                "max_pu_voltage": max_pu_voltage,
                "max_voltage_node_id": max_voltage_node_id,
                "min_pu_voltage": min_pu_voltage,
                "min_voltage_node_id": min_voltage_node_id,
            })

        summary_df = pd.DataFrame(summary_data).set_index("timestamp")
        return summary_df

    def aggregate_line_loss_and_loading_results(
        self,
        power_flow_results: BatchResult
    ) -> pd.DataFrame:
        """
        Aggregates line loss and loading results into a summary table.

        The table includes energy loss (kWh), maximum and minimum loading (p.u.)
        for each line across the entire timeline, along with their respective timestamps.

        Args:
            power_flow_results (BatchResult): The results from time-series power flow.

        Returns:
            pd.DataFrame: A table with aggregated line results.
                          Index: 'line_id'
                          Columns: 'energy_loss_kwh', 'max_loading_pu',
                                   'max_loading_timestamp', 'min_loading_pu',
                                   'min_loading_timestamp'.
        """
        line_losses_pu = power_flow_results.get_data("line", "p_loss_pu")
        line_loadings_pu = power_flow_results.get_data("line", "loading_pu")

        # Get all unique line IDs across all timestamps
        all_line_ids = set()
        for ts_data in line_losses_pu.values():
            all_line_ids.update(ts_data.keys())

        timestamps = sorted([pd.to_datetime(ts) for ts in line_losses_pu.keys()])
        if not timestamps:
            return pd.DataFrame(columns=[
                'energy_loss_kwh', 'max_loading_pu', 'max_loading_timestamp',
                'min_loading_pu', 'min_loading_timestamp'
            ])

        # Calculate time intervals in hours for integration
        # Assuming timestamps are regular, if not, calculate for each step
        time_intervals_hours = [(timestamps[i] - timestamps[i-1]).total_seconds() / 3600
                                for i in range(1, len(timestamps))]
        # If there's only one timestamp, interval is 0, so energy loss is 0.
        # If there are multiple, the first interval is often assumed from 0 or previous point.
        # For trapezoidal rule, we need intervals between points.

        summary_data = []
        for line_id in sorted(list(all_line_ids)):
            losses_for_line = []
            loadings_for_line = []
            loading_timestamps = []

            for ts_str in sorted(line_losses_pu.keys()): # Iterate through timestamps in order
                ts = pd.to_datetime(ts_str)
                # Get loss and loading for the current line at the current timestamp
                loss_pu = line_losses_pu.get(ts_str, {}).get(line_id, 0.0) # Default to 0 if not present
                loading_pu = line_loadings_pu.get(ts_str, {}).get(line_id, 0.0) # Default to 0 if not present

                losses_for_line.append(loss_pu)
                loadings_for_line.append(loading_pu)
                loading_timestamps.append(ts)

            # Calculate energy loss using trapezoidal rule
            # Convert p.u. power loss to kW loss first
            losses_kw = [loss_pu * self._base_power_mva * 1000 for loss_pu in losses_for_line] # MVA * 1000 = kVA, assuming unity power factor for power loss
            energy_loss_kwh = trapezoidal_integral(losses_kw, loading_timestamps)

            # Calculate max/min loading and their timestamps
            if loadings_for_line:
                max_loading_pu = max(loadings_for_line)
                min_loading_pu = min(loadings_for_line)

                # Find timestamps for max/min loading
                max_loading_timestamp = loading_timestamps[np.argmax(loadings_for_line)]
                min_loading_timestamp = loading_timestamps[np.argmin(loadings_for_line)]
            else:
                max_loading_pu = np.nan
                min_loading_pu = np.nan
                max_loading_timestamp = pd.NaT
                min_loading_timestamp = pd.NaT


            summary_data.append({
                "line_id": line_id,
                "energy_loss_kwh": energy_loss_kwh,
                "max_loading_pu": max_loading_pu,
                "max_loading_timestamp": max_loading_timestamp,
                "min_loading_pu": min_loading_pu,
                "min_loading_timestamp": min_loading_timestamp,
            })

        summary_df = pd.DataFrame(summary_data).set_index("line_id")
        return summary_df


def trapezoidal_integral(y_values: list[float], x_values: list[pd.Timestamp]) -> float:
    """
    Calculates the discrete numerical integral using the Trapezoidal rule.

    Args:
        y_values (list[float]): List of function values (e.g., power in kW).
        x_values (list[pd.Timestamp]): List of corresponding timestamps.

    Returns:
        float: The calculated integral value (e.g., energy in kWh).
    """
    if len(y_values) < 2 or len(x_values) < 2:
        return 0.0 # Cannot apply trapezoidal rule with less than 2 points

    integral = 0.0
    for i in range(len(y_values) - 1):
        # Calculate the width of the trapezoid (time difference in hours)
        delta_x_hours = (x_values[i+1] - x_values[i]).total_seconds() / 3600.0
        # Sum of parallel sides (y_values)
        sum_y = y_values[i] + y_values[i+1]
        # Area of trapezoid: 0.5 * (sum of parallel sides) * height
        integral += 0.5 * sum_y * delta_x_hours
    return integral

# Note: The `convert_power_to_energy_kwh` function is now integrated directly
# into the `aggregate_line_loss_and_loading_results` method for clarity,
# as the conversion factor depends on the base power of the grid.